# 04 — Plots & Visualization

Run `02_Run_Baselines.ipynb` and `03_Train_RL.ipynb` first.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os

from functions.baseline import compute_all_metrics

RESULTS_DIR = "../Results"
SAVE_PLOTS = True

COLORS = {
    "QQQ Buy-and-Hold":     "#636EFA",
    "Equal-Weight Monthly": "#EF553B",
    "Inverse Volatility":   "#00CC96",
    "Momentum Top-20%":     "#AB63FA",
    "Supervised + MVO":     "#FFA15A",
    "RL Agent":             "#FF6692",
    "QQQ (matched)":        "#636EFA",
}

def get_color(name):
    for key, color in COLORS.items():
        if key in name:
            return color
    return "#19D3F3"

## 1. Load Data

In [2]:
# Baselines (full period — may have different date range than RL)
bl_equity = pd.read_csv(f"{RESULTS_DIR}/equity_curves_full.csv", index_col=0, parse_dates=True)
bl_metrics = pd.read_csv(f"{RESULTS_DIR}/performance_metrics_full.csv", index_col=0)
print(f"Baselines: {list(bl_equity.columns)}")

# RL (stitched OOS equity)
rl_available = os.path.exists(f"{RESULTS_DIR}/rl_equity_oos.csv")
if rl_available:
    rl_equity = pd.read_csv(f"{RESULTS_DIR}/rl_equity_oos.csv", index_col=0, parse_dates=True)
    rl_returns = pd.read_csv(f"{RESULTS_DIR}/rl_daily_returns_oos.csv", index_col=0, parse_dates=True)
    rl_metrics = pd.read_csv(f"{RESULTS_DIR}/rl_performance_metrics.csv", index_col=0)
    print(f"RL OOS: {rl_equity.index[0]} → {rl_equity.index[-1]} ({len(rl_equity)} days)")
    display(rl_metrics)
else:
    print("RL not trained yet. Run 03_Train_RL.ipynb first.")

Baselines: ['QQQ Buy-and-Hold', 'Equal-Weight Monthly', 'Inverse Volatility', 'Momentum Top-20%', 'Supervised + MVO']
RL OOS: 2023-05-08 00:00:00 → 2026-02-04 00:00:00 (650 days)


,Absolute Return (%),ARC (%),ASD (%),Max Drawdown (%),MLD (years),IR1,IR2,Sharpe,Sortino,Calmar,N Days,Avg Daily Turnover (%)
RL Agent,52.5185,17.6200,17.0709,15.7139,0.2143,1.0322,1.1574,1.0436,1.6017,1.1213,650.0,1.6502
QQQ,85.7581,27.9592,19.5031,18.2037,0.3769,1.4336,2.2019,1.3284,1.8846,1.5359,650.0,NaN


## 2. RL Agent vs QQQ Buy-and-Hold (same period)

Recalculate QQQ B&H starting from the exact first day of the RL OOS test set,
so both curves start at 1.0 on the same date. This is the **fair** comparison.

In [3]:
if rl_available:
    # Recalculate QQQ equity from RL returns (already aligned)
    qqq_matched = (1 + rl_returns["QQQ"]).cumprod()
    rl_eq = (1 + rl_returns["RL Agent"]).cumprod()

    # Plot
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rl_eq.index, y=rl_eq, name="RL Agent (OOS)", mode="lines",
        line=dict(color=COLORS["RL Agent"], width=3.0)))
    fig.add_trace(go.Scatter(
        x=qqq_matched.index, y=qqq_matched, name="QQQ Buy-and-Hold", mode="lines",
        line=dict(color=COLORS["QQQ (matched)"], width=2.0, dash="dash"), opacity=0.8))

    fig.update_layout(
        title="Equity Curves — RL Agent vs QQQ (same OOS period, both start at $1)",
        xaxis_title="Date", yaxis_title="Portfolio Value ($1 start)",
        template="plotly_white", height=550,
        legend=dict(x=0.02, y=0.98), hovermode="x unified")
    if SAVE_PLOTS: fig.write_html(f"{RESULTS_DIR}/plot_equity_rl_vs_qqq.html")
    fig.show()

    # Metrics table: RL vs QQQ on SAME period
    rl_eq_arr = np.array([1.0] + list(rl_eq.values))
    qqq_eq_arr = np.array([1.0] + list(qqq_matched.values))
    rl_m = compute_all_metrics(rl_eq_arr)
    qqq_m = compute_all_metrics(qqq_eq_arr)

    fair_compare = pd.DataFrame({"RL Agent": rl_m, "QQQ Buy-and-Hold": qqq_m}).T
    print(f"\nFair comparison ({rl_eq.index[0].date()} → {rl_eq.index[-1].date()}):")
    display(fair_compare[["ARC (%)", "ASD (%)", "Max Drawdown (%)", "Sharpe", "Sortino", "Calmar", "IR1", "IR2"]])


Fair comparison (2023-05-08 → 2026-02-04):


,ARC (%),ASD (%),Max Drawdown (%),Sharpe,Sortino,Calmar,IR1,IR2
RL Agent,17.6200,17.0709,15.7139,1.0436,1.6017,1.1213,1.0322,1.1574
QQQ Buy-and-Hold,27.9592,19.5031,18.2037,1.3284,1.8846,1.5359,1.4336,2.2019


## 3. All Strategies Equity Chart

Baselines on their full period, RL on its OOS period.
Overlapping dates = fair comparison zone.

In [4]:
fig = go.Figure()

# Baselines
for col in bl_equity.columns:
    fig.add_trace(go.Scatter(
        x=bl_equity.index, y=bl_equity[col], name=col, mode="lines",
        line=dict(color=get_color(col), width=2.0 if "QQQ" in col else 1.2), opacity=0.7))

# RL Agent
if rl_available:
    fig.add_trace(go.Scatter(
        x=rl_equity.index, y=rl_equity["RL Agent"],
        name="RL Agent (OOS)", mode="lines",
        line=dict(color=COLORS["RL Agent"], width=3.0)))

fig.update_layout(
    title="Equity Curves — All Strategies",
    xaxis_title="Date", yaxis_title="Portfolio Value ($1 start)",
    template="plotly_white", height=600,
    legend=dict(x=0.02, y=0.98), hovermode="x unified")
if SAVE_PLOTS: fig.write_html(f"{RESULTS_DIR}/plot_equity_all.html")
fig.show()

## 4. Metrics Comparison — RL vs All Baselines

Note: Baselines were run on their full period (more data), RL on OOS only (less data).
This is not a perfectly fair comparison — the fair one is Section 2 above.

In [5]:
if rl_available:
    rl_row = rl_metrics.loc[["RL Agent"]]
    combined = pd.concat([bl_metrics, rl_row]).sort_values("IR2", ascending=False)
    print("All strategies (baselines = full period, RL = OOS only):")
    display(combined[["ARC (%)", "ASD (%)", "Max Drawdown (%)", "Sharpe", "Sortino", "IR1", "IR2", "Avg Daily Turnover (%)"]])

All strategies (baselines = full period, RL = OOS only):


,ARC (%),ASD (%),Max Drawdown (%),Sharpe,Sortino,IR1,IR2,Avg Daily Turnover (%)
RL Agent,17.6200,17.0709,15.7139,1.0436,1.6017,1.0322,1.1574,1.6502
QQQ Buy-and-Hold,14.9126,22.4795,35.1187,NaN,NaN,0.6634,0.2817,0.0000
Inverse Volatility,8.0979,20.3985,32.2320,NaN,NaN,0.3970,0.0997,1.5647
Equal-Weight Monthly,8.0377,20.2418,32.5878,NaN,NaN,0.3971,0.0979,0.3664
Momentum Top-20%,5.3805,22.5546,38.6482,NaN,NaN,0.2386,0.0332,34.7167
Supervised + MVO,-3.7547,22.9542,44.9105,NaN,NaN,0.0000,0.0000,68.1085


## 5. LaTeX Tables

In [ ]:
cols = ["ARC (%)", "ASD (%)", "Max Drawdown (%)", "MLD (years)", "Sharpe", "Sortino", "Calmar", "IR1", "IR2"]

if rl_available:
    # Table 1: Fair comparison (RL vs QQQ, same period)
    print("--- Table 1: RL Agent vs QQQ Buy-and-Hold (same OOS period) ---")
    fair_cols = [c for c in cols if c in fair_compare.columns]
    print(fair_compare[fair_cols].to_latex(float_format="%.3f",
        caption="RL Agent vs QQQ Buy-and-Hold (Same OOS Period, 5 bps TC)",
        label="tab:rl_vs_qqq"))

    # Table 2: All strategies
    print("\n--- Table 2: All Strategies ---")
    combined = pd.concat([bl_metrics, rl_metrics.loc[["RL Agent"]]]).sort_values("IR2", ascending=False)
    c_cols = [c for c in cols if c in combined.columns]
    print(combined[c_cols].to_latex(float_format="%.3f",
        caption="All Strategies Comparison", label="tab:all"))